# 4. Target-Limited Stratified N-Gram Trends

This notebook studies how a target-limited set of exact n-grams `V_n` and harmonic n-grams `H_n` vary across metadata strata: year, decade, and genre.

Notebook 3 owns the DuckDB build. This notebook adds trend-analysis tables to that same database and then queries them.


## Pipeline Role

This notebook is a lightweight trend tracker. It counts a fixed set of globally common exact and harmonic targets across strata such as year, decade, and genre.

That design is intentional: it keeps exploratory trend tables small and readable. It is **not** the full document-term matrix needed for TF-IDF or stop-gram discovery. Notebook 7 builds that broader artifact.

## Analysis Plan

Notebook 3 stores global vocabularies and the exact-to-harmonic map, but not per-song n-gram occurrence rows. That is good for storage, but it means stratified trends cannot be computed from `exact_ngrams` alone.

This notebook therefore re-streams the raw songs once and counts selected trend targets by stratum. By default it tracks the top global exact and harmonic n-grams for each `n`. Harmonic membership is read from notebook 3's `exact_ngrams` table, so notebook 4 uses the same `V_n -> H_n` mapping already validated in notebook 3.

The intended comparison is:

- exact trend: how a literal progression in `V_n` moves across strata.
- harmonic trend: how the corresponding transposition-normalized class in `H_n` moves across strata.
- comparison trend: whether exact progressions behave like their harmonic class, or whether spelling/key-specific representatives have their own stratified behavior.


In [1]:
from pathlib import Path
import importlib
import sys

import duckdb
import pandas as pd

CWD = Path.cwd()
ROOT = CWD.parent if (CWD / "utils").exists() else CWD
NOTEBOOK_DIR = ROOT / "notebooks"

# Force this repo's notebook utilities to win over any stale or external `utils` package.
sys.path = [p for p in sys.path if p != str(NOTEBOOK_DIR)]
sys.path.insert(0, str(NOTEBOOK_DIR))
for module_name in list(sys.modules):
    if module_name == "utils" or module_name.startswith("utils."):
        del sys.modules[module_name]

from utils import duckdb_store as ds
from utils import trend_analysis as ta

ds = importlib.reload(ds)
ta = importlib.reload(ta)

expected_files = {
    "duckdb_store": (NOTEBOOK_DIR / "utils" / "duckdb_store.py").resolve(),
    "trend_analysis": (NOTEBOOK_DIR / "utils" / "trend_analysis.py").resolve(),
}
loaded_files = {
    "duckdb_store": Path(ds.__file__).resolve(),
    "trend_analysis": Path(ta.__file__).resolve(),
}
assert loaded_files == expected_files, f"Imported wrong utility module(s): {loaded_files}; expected {expected_files}"

{
    "python": sys.executable,
    "duckdb_version": duckdb.__version__,
    **{name: str(path) for name, path in loaded_files.items()},
}

{'python': '/usr/local/bin/python3',
 'duckdb_version': '1.5.2',
 'duckdb_store': '/Users/juansalinas/Documents/GitHub/harmonic-trends/notebooks/utils/duckdb_store.py',
 'trend_analysis': '/Users/juansalinas/Documents/GitHub/harmonic-trends/notebooks/utils/trend_analysis.py'}

In [2]:
RAW_PATH = ROOT / "data" / "raw" / "chordonomicon_v2.csv"
DB_PATH = ROOT / "data" / "processed" / "harmonic_trends.duckdb"

NS = tuple(range(3, 9))
TOP_K_PER_N = 100
STRATA = ("year", "decade", "main_genre")
CHUNKSIZE = 25_000
MAX_ROWS = None
SHOW_PROGRESS = True
OVERWRITE = True

assert RAW_PATH.exists(), RAW_PATH
assert DB_PATH.exists(), DB_PATH
ROOT, DB_PATH

(PosixPath('/Users/juansalinas/Documents/GitHub/harmonic-trends'),
 PosixPath('/Users/juansalinas/Documents/GitHub/harmonic-trends/data/processed/harmonic_trends.duckdb'))

## Preflight

This checks whether notebook 2 has built the full DuckDB database. If the counts look like a smoke run, rerun notebook 2 with `MAX_ROWS = None` before using this notebook for real analysis.

Expected full-corpus signs after notebook 3:

- `song_ngram_summary` should have hundreds of thousands of rows.
- `exact_ngrams` should have tens of millions of rows for `n = 3..8`.
- `frequency_validation.counts_match` should be true for every `n`.


In [3]:
con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    ds.configure_connection(con)
    tables = con.execute("SHOW TABLES").fetchdf()
    validation = con.execute("SELECT * FROM frequency_validation ORDER BY n").fetchdf()
    sizes = con.execute(
        """
        SELECT 'exact_ngrams' AS table_name, COUNT(*) AS rows FROM exact_ngrams
        UNION ALL
        SELECT 'harmonic_ngrams' AS table_name, COUNT(*) AS rows FROM harmonic_ngrams
        UNION ALL
        SELECT 'song_ngram_summary' AS table_name, COUNT(*) AS rows FROM song_ngram_summary
        """
    ).fetchdf()
finally:
    con.close()

sizes, validation

(           table_name      rows
 0        exact_ngrams  34464863
 1     harmonic_ngrams  20696047
 2  song_ngram_summary    679807,
    n   V_count   H_count  V_frequency_sum  H_frequency_sum  counts_match
 0  3  50635145  50635145              1.0              1.0          True
 1  4  49955522  49955522              1.0              1.0          True
 2  5  49276039  49276039              1.0              1.0          True
 3  6  48596846  48596846              1.0              1.0          True
 4  7  47918358  47918358              1.0              1.0          True
 5  8  47240720  47240720              1.0              1.0          True)

In [4]:
song_rows = int(sizes.loc[sizes["table_name"].eq("song_ngram_summary"), "rows"].iloc[0])
exact_rows = int(sizes.loc[sizes["table_name"].eq("exact_ngrams"), "rows"].iloc[0])
validation_ok = bool(validation["counts_match"].all())

assert song_rows > 100_000, (
    f"song_ngram_summary has {song_rows:,} rows. This looks like a smoke/sample database; "
    "rerun notebook 2 with MAX_ROWS = None before trend analysis."
)
assert exact_rows > 10_000_000, (
    f"exact_ngrams has {exact_rows:,} rows. This is too small for the full n=3..8 corpus; "
    "rerun notebook 2 with MAX_ROWS = None."
)
assert validation_ok, "frequency_validation failed; inspect notebook 3 before running trend analysis."

"preflight passed"


'preflight passed'

## Add Song Metadata

The trend counts need song metadata for stratification. This table stores only compact metadata fields, not the chord strings.

In [5]:
ta.create_song_metadata_table(
    db_path=DB_PATH,
    raw_path=RAW_PATH,
    overwrite=OVERWRITE,
)

PosixPath('/Users/juansalinas/Documents/GitHub/harmonic-trends/data/processed/harmonic_trends.duckdb')

In [6]:
con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    ds.configure_connection(con)
    metadata_summary = con.execute(
        """
        SELECT
            COUNT(*) AS songs,
            COUNT(release_year) AS songs_with_year,
            COUNT(decade) AS songs_with_decade,
            COUNT(main_genre) AS songs_with_main_genre,
            COUNT(rock_genre) AS songs_with_rock_genre
        FROM song_metadata
        """
    ).fetchdf()
    by_decade = con.execute(
        """
        SELECT decade, COUNT(*) AS songs
        FROM song_metadata
        WHERE decade IS NOT NULL
        GROUP BY decade
        ORDER BY decade
        """
    ).fetchdf()
    by_genre = con.execute(
        """
        SELECT main_genre, COUNT(*) AS songs
        FROM song_metadata
        WHERE main_genre IS NOT NULL
        GROUP BY main_genre
        ORDER BY songs DESC
        LIMIT 20
        """
    ).fetchdf()
finally:
    con.close()

metadata_summary, by_decade, by_genre

(    songs  songs_with_year  songs_with_decade  songs_with_main_genre  \
 0  679807           422181             422181                 352111   
 
    songs_with_rock_genre  
 0                 145218  ,
     decade   songs
 0     1890      28
 1     1900       5
 2     1910       2
 3     1920      49
 4     1930      83
 5     1940     148
 6     1950    1748
 7     1960    9488
 8     1970   16110
 9     1980   19411
 10    1990   42474
 11    2000   93549
 12    2010  185492
 13    2020   53594,
      main_genre  songs
 0           pop  85185
 1          rock  67238
 2       country  53306
 3   alternative  47252
 4      pop rock  39557
 5          punk  16066
 6         metal  11315
 7           rap  11186
 8          soul   7350
 9          jazz   7001
 10       reggae   3841
 11   electronic   2814)

## Build Stratified Trend Tables

This is the main counting step. It scans raw songs and writes these tables/views into DuckDB:

- `trend_exact_counts`: counts of selected exact `V_n` targets by stratum.
- `trend_harmonic_counts`: counts of selected harmonic `H_n` targets by stratum, using notebook 3's exact-to-harmonic map.
- `trend_stratum_totals`: denominator windows by stratum and `n`.
- `trend_stratum_songs`: song counts by stratum.
- `trend_exact`: exact counts plus normalized frequencies.
- `trend_harmonic`: harmonic counts plus normalized frequencies.
- `trend_comparison`: exact frequencies joined to their harmonic class frequencies.

Progress is shown during target selection, raw-song scanning, and DuckDB table writes. For development, set `MAX_ROWS = 50_000` or lower. For real output, keep `MAX_ROWS = None`.

In [7]:
trend_summary = ta.build_stratified_target_trends(
    db_path=DB_PATH,
    raw_path=RAW_PATH,
    ns=NS,
    top_k_per_n=TOP_K_PER_N,
    strata=STRATA,
    chunksize=CHUNKSIZE,
    max_rows=MAX_ROWS,
    show_progress=SHOW_PROGRESS,
    overwrite=OVERWRITE,
)

trend_summary

Selecting top global exact and harmonic targets from DuckDB...


Counting stratified trend targets: 28chunk [1:21:58, 175.66s/chunk, rows=679807, V_hits=7.45e+7, H_hits=1.66e+8]
Scanning songs: 100%|██████████| 679807/679807 [1:21:58<00:00, 138.22song/s] 


Writing trend tables to DuckDB (53,351 exact rows, 55,839 harmonic rows)...
Stratified trend tables complete.


{'exact_trend_rows': 53351,
 'harmonic_trend_rows': 55839,
 'stratum_total_rows': 810,
 'stratum_song_rows': 135}

## Inspect Top Trends

Start by checking which exact and harmonic n-grams dominate each stratum. Frequencies are normalized by total windows for that stratum and `n`, so decade and genre strata are comparable despite different corpus sizes.

In [8]:
con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    ds.configure_connection(con)
    top_exact_by_decade = con.execute(
        """
        SELECT stratum_value AS decade, n, exact_ngram_id, ngram, count, total_windows, frequency
        FROM trend_exact
        WHERE stratum_type = 'decade'
        QUALIFY ROW_NUMBER() OVER (PARTITION BY stratum_value, n ORDER BY frequency DESC, count DESC) <= 5
        ORDER BY decade, n, frequency DESC
        """
    ).fetchdf()

    top_harmonic_by_decade = con.execute(
        """
        SELECT stratum_value AS decade, n, harmonic_id, example_ngram, count, total_windows, frequency
        FROM trend_harmonic
        WHERE stratum_type = 'decade'
        QUALIFY ROW_NUMBER() OVER (PARTITION BY stratum_value, n ORDER BY frequency DESC, count DESC) <= 5
        ORDER BY decade, n, frequency DESC
        """
    ).fetchdf()
finally:
    con.close()

top_exact_by_decade, top_harmonic_by_decade

(    decade  n      exact_ngram_id                  ngram  count  \
 0     1890  3  3_d6f9f19a01012d12                  G D C    124   
 1     1890  3  3_6adafd20068442f4                  C G D    102   
 2     1890  3  3_76b44d3da91e62d1                  D C G     91   
 3     1890  3  3_fe4dbe8dc705a25b                  D G D     76   
 4     1890  3  3_6458674f98ac0103                  G D G     35   
 ..     ... ..                 ...                    ...    ...   
 375   2020  8  8_a993930203ba4d88  C G Amin F C G Amin F   9426   
 376   2020  8  8_bc1b0ab55a7a9e0e  Amin F C G Amin F C G   9414   
 377   2020  8  8_29b2583fd1aa8f8d  F C G Amin F C G Amin   9137   
 378   2020  8  8_f1343192a1a4d3db  G Amin F C G Amin F C   9110   
 379   2020  8  8_7c63fcf4e05587a0  Emin C G D Emin C G D   7816   
 
      total_windows  frequency  
 0             2429   0.051050  
 1             2429   0.041993  
 2             2429   0.037464  
 3             2429   0.031289  
 4             24

## Compare Exact vs Harmonic Trends

This comparison asks: when an exact n-gram rises or falls in a stratum, is the whole harmonic class moving with it, or is that exact spelling/progression behaving differently from its transposition-equivalent class?

In [9]:
con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    ds.configure_connection(con)
    comparison_by_decade = con.execute(
        """
        SELECT
            stratum_value AS decade,
            n,
            ngram,
            exact_ngram_id,
            harmonic_id,
            exact_count,
            harmonic_count,
            exact_frequency,
            harmonic_frequency,
            harmonic_to_exact_frequency_ratio
        FROM trend_comparison
        WHERE stratum_type = 'decade'
        ORDER BY n, decade, exact_frequency DESC
        LIMIT 100
        """
    ).fetchdf()
finally:
    con.close()

comparison_by_decade

,decade,n,ngram,exact_ngram_id,harmonic_id,exact_count,harmonic_count,exact_frequency,harmonic_frequency,harmonic_to_exact_frequency_ratio
0,1890,3,G D C,3_d6f9f19a01012d12,H3_2bd39a67fe3535a8,124,167,0.051050,0.068753,1.346774
1,1890,3,C G D,3_6adafd20068442f4,H3_4e99eecf61b0edec,102,144,0.041993,0.059284,1.411765
2,1890,3,D C G,3_76b44d3da91e62d1,H3_0859351c7b2e5686,91,130,0.037464,0.053520,1.428571
3,1890,3,D G D,3_fe4dbe8dc705a25b,H3_ede3c4f53675bbb0,76,135,0.031289,0.055578,1.776316
4,1890,3,G D G,3_6458674f98ac0103,H3_35d6bfda85b78e3b,35,81,0.014409,0.033347,2.314286
...,...,...,...,...,...,...,...,...,...,...
95,1920,3,A D A,3_2ed1bbbae7af8d62,H3_ede3c4f53675bbb0,2,137,0.000617,0.042232,68.500000
96,1920,3,D G Emin,3_fe6b07e31d44c42d,H3_f6794f69165ef172,1,3,0.000308,0.000925,3.000000
97,1920,3,Emin D G,3_cb7a608e02995caa,H3_3f6165bb1739c85d,1,4,0.000308,0.001233,4.000000
98,1920,3,Emin G D,3_cc8d80d0736ff3ca,H3_4e2fdb872fee09bf,1,1,0.000308,0.000308,1.000000


## Year and Genre Slices

These are noisier than decade-level trends. For year-level plots or tables, usually filter to strata with enough denominator windows first.

In [10]:
MIN_WINDOWS = 10_000
N = 3

con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    ds.configure_connection(con)
    year_trends = con.execute(
        """
        SELECT y.stratum_value AS year, y.n, y.harmonic_id, y.example_ngram, y.count, y.total_windows, y.frequency
        FROM trend_harmonic y
        WHERE y.stratum_type = 'year'
          AND y.n = ?
          AND y.total_windows >= ?
        QUALIFY ROW_NUMBER() OVER (PARTITION BY y.stratum_value ORDER BY y.frequency DESC, y.count DESC) <= 10
        ORDER BY year, frequency DESC
        """,
        [N, MIN_WINDOWS],
    ).fetchdf()

    genre_trends = con.execute(
        """
        SELECT g.stratum_value AS main_genre, g.n, g.harmonic_id, g.example_ngram, g.count, g.total_windows, g.frequency
        FROM trend_harmonic g
        WHERE g.stratum_type = 'main_genre'
          AND g.n = ?
          AND g.total_windows >= ?
        QUALIFY ROW_NUMBER() OVER (PARTITION BY g.stratum_value ORDER BY g.frequency DESC, g.count DESC) <= 10
        ORDER BY main_genre, frequency DESC
        """,
        [N, MIN_WINDOWS],
    ).fetchdf()
finally:
    con.close()

year_trends, genre_trends

(     year  n          harmonic_id example_ngram  count  total_windows  \
 0    1956  3  H3_ede3c4f53675bbb0         G C G    861          12221   
 1    1956  3  H3_35d6bfda85b78e3b         C G C    738          12221   
 2    1956  3  H3_614d8a0823c338e1         D G C    341          12221   
 3    1956  3  H3_4e99eecf61b0edec         C G D    324          12221   
 4    1956  3  H3_3569c7a41696606d        C G7 C    186          12221   
 ..    ... ..                  ...           ...    ...            ...   
 675  2023  3  H3_614d8a0823c338e1         D G C  15022         655586   
 676  2023  3  H3_cb0f464e362cf3bc      Emin C G  14003         655586   
 677  2023  3  H3_0859351c7b2e5686         D C G  13491         655586   
 678  2023  3  H3_950fb5de7bd72a8d      G Amin F  12787         655586   
 679  2023  3  H3_2bd39a67fe3535a8         G D C  12080         655586   
 
      frequency  
 0     0.070452  
 1     0.060388  
 2     0.027903  
 3     0.026512  
 4     0.015220  
 .

## Notes for the Next Iteration

The current notebook tracks top global targets. That is the right first analysis route because it avoids materializing every `V_n` by every stratum. If we later need exhaustive stratified vocabularies, we should build them as chunked DuckDB tables with intermediate aggregation, not as Pandas dictionaries.

Good next questions:

- Which harmonic classes have the largest decade-to-decade frequency change?
- Which exact n-grams are unusually genre-specific relative to their harmonic class?
- Which harmonic classes are stable globally but split into different exact representatives across genres?

## Handoff

Use these tables for top-target trend inspection and exact-vs-harmonic representative comparisons.

For corpus-linguistic statistics such as document frequency, IDF, stop-grams, and keyness, use the broader document-term tables built in notebook 7.